# Data Integration and Data Transformation for Data Mining

In [1]:
import pandas as pd
import numpy as np

#### Sample Datasets for Data Integration

In [2]:
data_1 = {
    'ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Kusbhoo', 'Aryan', 'Sujal', 'Ganesh', 'Astha', 'Aditi'],
    'Age': [25, 30, 35, 40, 30, 40, 50, 18, 80, 102]
}

data_2 = {
    'ID': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'Gender': ['F', 'M', 'M', 'F','F', 'M', 'M', 'F', 'M','F'],
    'Salary': [70000, 80000, 50000, 60000, 30000, 35000, 12000, 50000, 40000, 50000]
}

df1 = pd.DataFrame(data_1)
df2 = pd.DataFrame(data_2)

### Data Integration
#### Tight Coupling (Join datasets on a common key)

In [3]:
tight_coupling = pd.merge(df1, df2, on='ID', how='inner')
print("Tight Coupling Result:\n", tight_coupling)

Tight Coupling Result:
    ID     Name  Age Gender  Salary
0   3  Charlie   35      F   70000
1   4    David   40      M   80000
2   5  Kusbhoo   30      M   50000
3   6    Aryan   40      F   60000
4   7    Sujal   50      F   30000
5   8   Ganesh   18      M   35000
6   9    Astha   80      M   12000
7  10    Aditi  102      F   50000



#### Loose Coupling (Concatenate datasets)


In [4]:
loose_coupling = pd.concat([df1.set_index('ID'), df2.set_index('ID')], axis=1).reset_index()
print("Loose Coupling Result:\n", loose_coupling)

Loose Coupling Result:
     ID     Name    Age Gender   Salary
0    1    Alice   25.0    NaN      NaN
1    2      Bob   30.0    NaN      NaN
2    3  Charlie   35.0      F  70000.0
3    4    David   40.0      M  80000.0
4    5  Kusbhoo   30.0      M  50000.0
5    6    Aryan   40.0      F  60000.0
6    7    Sujal   50.0      F  30000.0
7    8   Ganesh   18.0      M  35000.0
8    9    Astha   80.0      M  12000.0
9   10    Aditi  102.0      F  50000.0
10  11      NaN    NaN      M  40000.0
11  12      NaN    NaN      F  50000.0


### Data Transformation

#### Smoothing (Moving Average for Age)

In [5]:
loose_coupling['Smoothed_Age'] = loose_coupling['Age'].rolling(window=2, min_periods=1).mean()
print("\nSmoothing:\n", loose_coupling[['ID', 'Age', 'Smoothed_Age']])


Smoothing:
     ID    Age  Smoothed_Age
0    1   25.0          25.0
1    2   30.0          27.5
2    3   35.0          32.5
3    4   40.0          37.5
4    5   30.0          35.0
5    6   40.0          35.0
6    7   50.0          45.0
7    8   18.0          34.0
8    9   80.0          49.0
9   10  102.0          91.0
10  11    NaN         102.0
11  12    NaN           NaN


#### Aggregation (Summarizing Salary by Gender)

In [6]:
aggregation = loose_coupling.groupby('Gender')['Salary'].sum().reset_index()
print("\nAggregation:\n", aggregation)


Aggregation:
   Gender    Salary
0      F  260000.0
1      M  217000.0


#### Discretization (Binning Age into Categories)

In [7]:
bins = [0, 20, 30, 40, 50]
labels = ['Teen', 'Young Adult', 'Adult', 'Senior']
loose_coupling['Age_Group'] = pd.cut(loose_coupling['Age'], bins=bins, labels=labels)
print("\nDiscretization:\n", loose_coupling[['ID', 'Age', 'Age_Group']])


Discretization:
     ID    Age    Age_Group
0    1   25.0  Young Adult
1    2   30.0  Young Adult
2    3   35.0        Adult
3    4   40.0        Adult
4    5   30.0  Young Adult
5    6   40.0        Adult
6    7   50.0       Senior
7    8   18.0         Teen
8    9   80.0          NaN
9   10  102.0          NaN
10  11    NaN          NaN
11  12    NaN          NaN


#### Attribute Construction (Creating Age-Salary Ratio)

In [8]:
loose_coupling['Age_Salary_Ratio'] = loose_coupling['Age'] / loose_coupling['Salary']
print("\nAttribute Construction:\n", loose_coupling[['ID', 'Age', 'Salary', 'Age_Salary_Ratio']])


Attribute Construction:
     ID    Age   Salary  Age_Salary_Ratio
0    1   25.0      NaN               NaN
1    2   30.0      NaN               NaN
2    3   35.0  70000.0          0.000500
3    4   40.0  80000.0          0.000500
4    5   30.0  50000.0          0.000600
5    6   40.0  60000.0          0.000667
6    7   50.0  30000.0          0.001667
7    8   18.0  35000.0          0.000514
8    9   80.0  12000.0          0.006667
9   10  102.0  50000.0          0.002040
10  11    NaN  40000.0               NaN
11  12    NaN  50000.0               NaN


#### Generalization (Simplifying Salary into Ranges)

In [9]:
salary_bins = [0, 60000, 80000, 100000]
salary_labels = ['Low', 'Medium', 'High']
loose_coupling['Salary_Range'] = pd.cut(loose_coupling['Salary'], bins=salary_bins, labels=salary_labels)
print("\nGeneralization:\n", loose_coupling[['ID', 'Salary', 'Salary_Range']])


Generalization:
     ID   Salary Salary_Range
0    1      NaN          NaN
1    2      NaN          NaN
2    3  70000.0       Medium
3    4  80000.0       Medium
4    5  50000.0          Low
5    6  60000.0          Low
6    7  30000.0          Low
7    8  35000.0          Low
8    9  12000.0          Low
9   10  50000.0          Low
10  11  40000.0          Low
11  12  50000.0          Low


### Normalization

#### Min-Max Normalization

In [10]:
loose_coupling['Age_MinMax'] = (loose_coupling['Age'] - loose_coupling['Age'].min()) / (loose_coupling['Age'].max() - loose_coupling['Age'].min())
print("\nMin-Max Normalization:\n", loose_coupling[['ID', 'Age', 'Age_MinMax']])


Min-Max Normalization:
     ID    Age  Age_MinMax
0    1   25.0    0.083333
1    2   30.0    0.142857
2    3   35.0    0.202381
3    4   40.0    0.261905
4    5   30.0    0.142857
5    6   40.0    0.261905
6    7   50.0    0.380952
7    8   18.0    0.000000
8    9   80.0    0.738095
9   10  102.0    1.000000
10  11    NaN         NaN
11  12    NaN         NaN


#### Z-Score Normalization

In [11]:
loose_coupling['Age_ZScore'] = (loose_coupling['Age'] - loose_coupling['Age'].mean()) / loose_coupling['Age'].std()
print("\nZ-Score Normalization:\n", loose_coupling[['ID', 'Age', 'Age_ZScore']])


Z-Score Normalization:
     ID    Age  Age_ZScore
0    1   25.0   -0.760286
1    2   30.0   -0.570214
2    3   35.0   -0.380143
3    4   40.0   -0.190071
4    5   30.0   -0.570214
5    6   40.0   -0.190071
6    7   50.0    0.190071
7    8   18.0   -1.026386
8    9   80.0    1.330500
9   10  102.0    2.166815
10  11    NaN         NaN
11  12    NaN         NaN


#### Decimal Scaling

In [12]:
scaling_factor = 10 ** np.ceil(np.log10(loose_coupling['Age'].abs().max()))
loose_coupling['Age_DecimalScaling'] = loose_coupling['Age'] / scaling_factor
print("\nDecimal Scaling:\n", loose_coupling[['ID', 'Age', 'Age_DecimalScaling']])


Decimal Scaling:
     ID    Age  Age_DecimalScaling
0    1   25.0               0.025
1    2   30.0               0.030
2    3   35.0               0.035
3    4   40.0               0.040
4    5   30.0               0.030
5    6   40.0               0.040
6    7   50.0               0.050
7    8   18.0               0.018
8    9   80.0               0.080
9   10  102.0               0.102
10  11    NaN                 NaN
11  12    NaN                 NaN
